### Wrangling all predictors' predictions into a single dataframe
**only works for 40aa tiles (at least for the PADDLE consideration and compositional predictor)

1. Load seqfeat and all prediction files
2. Convert the string, bracket-enclosed predictions columns into floats or lists of floats
3. For 53-mer predictors, strip out 13aa synthetic TF sequence that was front-appended
4. For residue-level predictor ADPred: calculate two scores, the average of all 40 residue-level scores and binary classification based on Erijman et al. 2020
5. Create compositional predictions based on seqfeat file (official calculation: R+K-D-E)
6. Combine all 6 predictors into a single df
8. Save this df

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import ast

In [ ]:
seqfeat = pd.read_csv(r'C:\Users\music\Downloads\500k_AA_SS_seqfeats.csv')

pad = pd.read_csv(r"C:\Users\music\Downloads\500k_Design_AA_53_Paddle_preds_dask.csv")
tada=pd.read_csv(r"C:\Users\music\Downloads\500k_Design_AA_TADA_preds_dask (2).csv")
adh = pd.read_csv(r"C:\Users\music\Downloads\500k_Design_AA_ADHunter_preds_dask.csv")
adp = pd.read_csv(r"C:\Users\music\Downloads\500k_Design_AA_ADpred_preds_dask.csv")
adhe = pd.read_csv(r"C:\Users\music\Downloads\500k_Design_AA_53_ADhunter_ensemble_preds.csv")

originalinfo = pd.read_csv(r'C:\Users\music\OneDrive\Desktop\CompLab\SynAD\data\500k_original_2023_predictions.csv',index_col=0)

savefile = r'C:\Users\music\OneDrive\Desktop\CompLab\SynAD\Publication\GitHub\data_generated\500k_preds.csv'

In [3]:
pad["paddle_preds"] = pd.to_numeric(pad["paddle_preds"].str.strip("[]"), errors="coerce")
pad = pad[['sequence', 'paddle_preds']]
tada["tada_preds"] = pd.to_numeric(tada["tada_preds"].str.strip("[]"), errors="coerce")
tada = tada[['sequence', 'tada_preds']]
adh["adhunter_preds"]   = pd.to_numeric(adh["adhunter_preds"].str.strip("[]"), errors="coerce")
adh = adh[['sequence', 'adhunter_preds']]
adp["adpred_preds"] = adp["adpred_preds"].apply(lambda x: [float(i) for i in x.strip("[]").split(",")])
adp = adp[['sequence', 'adpred_preds']]
adhe["adhe_preds"] = pd.to_numeric(adhe["adhunter_ensemble_preds"].str.strip("[]"), errors="coerce")
adhe = adhe[['sequence', 'adhe_preds']]

In [4]:
# Convert 53-mer predictors back to 40-mer format\
pad['sequence'] = pad['sequence'].str.replace('SHLATAGSTSSAS', '', regex=False)

adhe = adhe[['sequence', 'adhe_preds']]
adhe['sequence'] = adhe['sequence'].str.replace('SHLATAGSTSSAS', '', regex=False)

In [5]:
# Calculate ADPred average of residue-level predictions and classification score
adp["adpred_ave"] = adp["adpred_preds"].apply(lambda x: sum(x) / len(x))
def adpred_classify(preds):
    """1 if there are >= 10 consecutive values >= 0.8, else 0."""
    run = 0
    for p in preds:
        run = run + 1 if p >= 0.8 else 0
        if run >= 10:
            return 1
    return 0
adp["adpred_classify"] = adp["adpred_preds"].apply(adpred_classify)
adp.drop(columns=['adpred_preds'], inplace=True)

In [6]:
# Compositional predictor
comppred = seqfeat.loc[:, ['AAseq', 'R', 'K', 'D', 'E', 'W', 'F', 'L']].copy()
comppred['charge'] = comppred['R'] + comppred['K'] - comppred['D'] - comppred['E']
comppred['WFL'] = comppred['W'] + comppred['F'] + comppred['L']

# vectorized masks (avoid Python `if` on a Series)
comp_mask = (comppred['charge'] >= -13) & (comppred['charge'] <= -8) & (comppred['WFL'] >= 6)
comppred['Comp'] = comp_mask.astype(int)

comppred.drop(columns=['R', 'K', 'D', 'E', 'W', 'F', 'L', 'charge', 'WFL'], inplace=True)
comppred.rename(columns={'AAseq': 'sequence'}, inplace=True)

In [7]:
dfs = {'adh': adh, 'tada': tada, 'pad': pad, 'adpred': adp, 'adhe': adhe, 'Comp': comppred}
ID_COLS = ['Unnamed: 0', 'name']   # per-row IDs, not meaningful once duplicates are dropped

allpreds = None
ok = True
for name, d in dfs.items():
    pred_cols = [c for c in d.columns if c not in ID_COLS + ['sequence']]

    # identical sequences must have identical predictions (compared as str so list columns work)
    n_conflict = (d[pred_cols].astype(str).groupby(d['sequence']).nunique() > 1).any(axis=1).sum()
    print(f"{name}: {len(d)} rows, {d['sequence'].nunique()} unique sequences, {n_conflict} with conflicting predictions")
    ok &= n_conflict == 0

    d = d.drop_duplicates('sequence')[['sequence'] + pred_cols]
    allpreds = d if allpreds is None else pd.merge(allpreds, d, on='sequence', how='inner', validate='one_to_one')

# one-to-one check: each df's unique sequences are exactly the sequences in allpreds
for name, d in dfs.items():
    same = set(d['sequence']) == set(allpreds['sequence'])
    print(f"{name} matches allpreds one-to-one: {same}")
    ok &= same

allpreds.rename(columns={'sequence': 'AAseq'}, inplace=True)
print(f"allpreds: {len(allpreds)} rows")

adh: 500000 rows, 427076 unique sequences, 0 with conflicting predictions
tada: 500000 rows, 427076 unique sequences, 0 with conflicting predictions
pad: 500000 rows, 427076 unique sequences, 0 with conflicting predictions
adpred: 500000 rows, 427076 unique sequences, 0 with conflicting predictions
adhe: 500000 rows, 427076 unique sequences, 0 with conflicting predictions
Comp: 500000 rows, 427076 unique sequences, 0 with conflicting predictions
adh matches allpreds one-to-one: True
tada matches allpreds one-to-one: True
pad matches allpreds one-to-one: True
adpred matches allpreds one-to-one: True
adhe matches allpreds one-to-one: True
Comp matches allpreds one-to-one: True
allpreds: 427076 rows


In [8]:
allpreds.rename(columns={'adhunter_preds':'ADhunterLite_Score','tada_preds':'TADA_Score','paddle_preds':'PADDLE_Score','adpred_ave':'ADPred_Score','adpred_classify':'ADPred','adhe_preds':'ADhunterEnsemble_Score'},inplace=True)
allpreds['ADhunterLite'] = (allpreds['ADhunterLite_Score'] > 100000).astype(int)
allpreds['TADA'] = (allpreds['TADA_Score'] > 0.4).astype(int)
allpreds['PADDLE'] = (allpreds['PADDLE_Score'] > 4).astype(int)
allpreds['ADhunterEnsemble'] = (allpreds['ADhunterEnsemble_Score'] > 0.4441).astype(int)

In [9]:
allpreds_originalinfo = pd.merge(allpreds, originalinfo, on='AAseq', how='right')
print(allpreds_originalinfo.shape)

(500000, 17)


In [10]:
allpreds_originalinfo.to_csv(savefile)